<a href="https://colab.research.google.com/github/saurav8835/Master_In_Artficial_Intelligence/blob/main/Principles_Of_AI/Week-2/Environments_Gymnasium_RationalAgents_Lab_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Rational Agents in Gymnasium

This lab is a hands-on companion to the **Rational Agents** lecture (AIMA Ch. 2). We use the [Gymnasium](https://github.com/Farama-Foundation/Gymnasium) library, which gives a clean, code-level version of the agent/environment picture from lecture.

Recall from lecture:

* An **agent** perceives its **environment** through **sensors** and acts on it through **actuators**.
* A **percept** is the agent's perceptual input at a given instant.
* A **rational agent** selects, for every percept sequence, the action that maximises the **expected value of its performance measure**.
* We specify a task environment using **PEAS**: Performance measure, Environment, Actuators, Sensors.
* Task environments can be classified as fully/partially observable, deterministic/stochastic, episodic/sequential, static/dynamic, discrete/continuous, and single-agent/multi-agent.

Gymnasium's API maps onto this directly:

| Rational agent concept | Gymnasium equivalent |
|---|---|
| Environment | the `env` object returned by `gym.make(...)` |
| Actuators / possible actions | `env.action_space` |
| Sensors / percept | `env.observation_space`, and the `obs` returned by `env.reset()` / `env.step()` |
| Performance measure (signal) | the `reward` returned by `env.step()`, accumulated over an episode |
| Agent program | whatever code decides `action` given `obs` |
| Episode ending | `terminated` (reached a terminal/goal/failure state) or `truncated` (e.g. time limit) |

Over the course of this lab we will look at three kinds of "agent program", from simplest to most sophisticated:

1. A **random agent** — ignores the observation completely. Not rational, just a baseline.
2. A **simple reflex agent** — a hand-written rule based only on the *current* percept.
3. A **simple (tabular) Q-learning agent** — learns, from experience, which action tends to lead to the best expected performance measure in each state. No neural networks needed — just a table of numbers.

## Looking at Gymnasium Environments

The centrepiece of Gymnasium is the **environment**, which defines the task the agent must solve. An environment does not need to be a game; it simply describes:

* **action space** — the actions the agent can take at each step (its actuators).
* **observation space** — the state of the (observable) part of the environment (its sensors/percept).

Gymnasium is installed with `pip`. We will only use the small, fast, classic-control and toy-text environments in this lab (no Atari, no external ROMs, no video rendering needed).

In [1]:
!pip install gymnasium


It is important to note that many Gymnasium environments specify that they are **not** deterministic even though they use random numbers to process actions (e.g. slippery ice, wind). Gymnasium lets us query these properties for any environment — useful for the task-environment classification exercise below.

In [8]:
import gymnasium as gym

def query_environment(name, **kwargs):
    env = gym.make(name, **kwargs)
    spec = gym.spec(name)
    env_id=env.spec.id if env.spec else 'None'
    print(f"Action Space: {env.action_space}")
    print(f"Observation Space: {env.observation_space}")
    print(f"Max Episode Steps: {spec.max_episode_steps}")
    print(f"Nondeterministic: {spec.nondeterministic}")
    print(f"Reward Threshold: {spec.reward_threshold}")

    print(f"\n=====================================")
    print(f" PEAS ANALYSIS FOR: {env_id}")
    print(f"=====================================")
    print(f"▶ ENVIRONMENT: Initialised {env_id}")

    # --- 2. ACTUATORS ---
    # Read directly from the action space to see what the agent can physically do
    print(f"\n▶ ACTUATORS (Action Space):")
    print(f"  Structure: {env.action_space}")
    if isinstance(env.action_space, gym.spaces.Discrete):
        print(f"  Available Actions: {list(range(env.action_space.n))}")

    # --- 3. SENSORS ---
    # Read from the observation space bounds to see what variables the agent perceives
    print(f"\n▶ SENSORS (Observation Space):")
    print(f"  Structure: {env.observation_space}")
    print(f"  Low Bounds:  {env.observation_space.low}")
    print(f"  High Bounds: {env.observation_space.high}")

    # --- 4. PERFORMANCE MEASURE (Limits & Rewards) ---
    # We inspect a single step simulation to look at the reward signal structure
    # and retrieve the built-in time truncation threshold.
    x,y=env.reset()
    next_obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    print(f"\n reset object x:{x},y:{y}, obs:{next_obs}")
    print(f"\n▶ PERFORMANCE MEASURE:")
    print(f"  Sample Step Reward Signal: {reward}")
    print(f"  Max Episode Step Limit (Truncation): {env.spec.max_episode_steps if env.spec else 'None'}")

    env.close()


We will look at the **MountainCar-v0** environment, which challenges an underpowered car to escape the valley between two mountains.

In [9]:
query_environment("MountainCar-v0")


Action Space: Discrete(3)
Observation Space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Max Episode Steps: 200
Nondeterministic: False
Reward Threshold: -110.0

 PEAS ANALYSIS FOR: MountainCar-v0
▶ ENVIRONMENT: Initialised MountainCar-v0

▶ ACTUATORS (Action Space):
  Structure: Discrete(3)
  Available Actions: [0, 1, 2]

▶ SENSORS (Observation Space):
  Structure: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
  Low Bounds:  [-1.2  -0.07]
  High Bounds: [0.6  0.07]

 reset object x:[-0.4396096  0.       ],y:{}, obs:[-0.4412329  -0.00162327]

▶ PERFORMANCE MEASURE:
  Sample Step Reward Signal: -1.0
  Max Episode Step Limit (Truncation): 200


This environment allows three distinct actions: accelerate forward, decelerate, or backward. The observation space contains two continuous (floating point) values — the position and velocity of the car. The car has 200 steps to escape each episode, and only receives a reward when it escapes the valley.

Lab Task 1: MountainCar-v0

Performance measure:
*   Receive a reward of -1.0 for every step taken
*   Minimize the total penalty by reaching the target flag.
*   The episode ends automatically after 200 steps
*   Target is a cumulative reward threshold of -110.0 or better

Environment:
*   valley between two mountains, underpowered engine

Actuators:
*   Discrete(3), accelerate forward, decelerate, or backward

Sensors:
*   Position, Velocity

In [10]:
query_environment("CartPole-v1")


Action Space: Discrete(2)
Observation Space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Max Episode Steps: 500
Nondeterministic: False
Reward Threshold: 475.0

 PEAS ANALYSIS FOR: CartPole-v1
▶ ENVIRONMENT: Initialised CartPole-v1

▶ ACTUATORS (Action Space):
  Structure: Discrete(2)
  Available Actions: [0, 1]

▶ SENSORS (Observation Space):
  Structure: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
  Low Bounds:  [-4.8               -inf -0.41887903        -inf]
  High Bounds: [4.8               inf 0.41887903        inf]

 reset object x:[ 0.00723203 -0.00153853 -0.03903417  0.00149517],y:{}, obs:[ 0.00720125 -0.19607957 -0.03900426  0.2816113 ]

▶ PERFORMANCE MEASURE:
  Sample Step Reward Signal: 1.0
  Max Episode Step Limit (Truncation): 500


The **CartPole-v1** environment challenges the agent to balance a pole on a moving cart. The observation space has 4 continuous numbers:

* Cart Position
* Cart Velocity
* Pole Angle
* Pole Angular Velocity

The agent can take two actions:

* Push cart to the left
* Push cart to the right

Each step the pole stays upright earns +1 reward, and the episode ends (`terminated`) if the pole falls too far or the cart leaves the track.

Lab Task 1: CartPole-v1

Performance measure:
*   +1 reward for staying upright.
*   The episode ends automatically after 500 steps

Environment:
*   Balance a pole on moving cart

Actuators:
*   Push Left, Push Right

Sensors:
*   Cart Position
*   Cart Velocity
*   Pole Angle
*   Pole Angular Velocity


### The `step` and `reset` functions

Both functions return several values:

* **observation** — an element of `observation_space`: the agent's percept.
* **reward** — the performance-measure signal for that step.
* **terminated** — whether the episode ended in a terminal state of the task itself (goal reached, pole fell, etc). If true, call `reset()`.
* **truncated** — whether the episode ended for a reason outside the task's own definition, most commonly a time limit. If true, call `reset()`.
* **info** — a dictionary of extra diagnostic information (not usually needed by the agent itself).

## Lab Task 1 — PEAS analysis (write-up)

For **each** of `MountainCar-v0` and `CartPole-v1`, add a text cell giving a full PEAS specification, in the same style used in lecture for the automated taxi driver / delivery drone:

* **Performance measure:** what should the agent be trying to maximise? (Hint: look at what `reward` is given for, and when the episode terminates vs. truncates.)
* **Environment:** what does the agent operate in/on?
* **Actuators:** what can the agent actually do? (Read off `env.action_space`.)
* **Sensors:** what can the agent perceive? (Read off `env.observation_space`.)

## Lab Task 2 — Classify the task environment

For each of the two environments, classify it along **all six** dimensions from lecture, with a one-sentence justification for each:

1. Fully observable vs. partially observable
2. Deterministic vs. stochastic (check the `Nondeterministic` flag above!)
3. Episodic vs. sequential
4. Static vs. dynamic
5. Discrete vs. continuous
6. Single-agent vs. multi-agent

Present your answers as a markdown table, one row per environment.

Lab Task 1: MountainCar-v0

Performance measure:
*   Receive a reward of -1.0 for every step taken
*   Minimize the total penalty by reaching the target flag.
*   The episode ends automatically after 200 steps
*   Target is a cumulative reward threshold of -110.0 or better

Environment:
*   valley between two mountains, underpowered engine

Actuators:
*   Discrete(3), accelerate forward, decelerate, or backward

Sensors:
*   Position, Velocity




Lab Task 1: CartPole-v1

```
# This is formatted as code
```



Lab Task 2:


## Random Agent vs. a Simple Reflex Agent

So far the only "agent" we've used is `env.action_space.sample()` — pure random action selection that ignores the observation entirely. Below we compare it against a **simple reflex agent** for `CartPole-v1`: an agent that selects its action based *only on the current percept* (the current pole angle), with no memory and no planning — exactly the "simple reflex agent" definition from lecture:

> *"A simple reflex agent is one that selects an action based only on the current percept. It ignores the rest of the percept history."*

In [ ]:
import gymnasium as gym
import numpy as np

def random_agent(env, obs):
    """Ignores the observation completely -- not rational."""
    return env.action_space.sample()

def simple_reflex_agent(env, obs):
    """CartPole observation: [cart position, cart velocity, pole angle, pole angular velocity].
    Simple reflex rule: push the cart in the direction the pole is already leaning,
    based only on the CURRENT percept (obs[2], the pole angle). No memory, no planning."""
    pole_angle = obs[2]
    return 1 if pole_angle > 0 else 0  # 1 = push right, 0 = push left

def run_episodes(policy, n_episodes=20, env_name="CartPole-v1", seed=42):
    env = gym.make(env_name)
    totals = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        done = False
        total_reward = 0.0
        while not done:
            action = policy(env, obs)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        totals.append(total_reward)
    env.close()
    return totals

random_rewards = run_episodes(random_agent)
reflex_rewards = run_episodes(simple_reflex_agent)

print(f"Random agent        -- mean reward: {np.mean(random_rewards):.1f}  (min {np.min(random_rewards):.0f}, max {np.max(random_rewards):.0f})")
print(f"Simple reflex agent -- mean reward: {np.mean(reflex_rewards):.1f}  (min {np.min(reflex_rewards):.0f}, max {np.max(reflex_rewards):.0f})")


Random agent        -- mean reward: 19.3  (min 11, max 41)
Simple reflex agent -- mean reward: 44.0  (min 32, max 56)


## Lab Task 3 — Build and evaluate your own reflex agent

1. Run the cell above and record the mean reward for both agents.
2. **Modify** `simple_reflex_agent` to also react to the pole's angular velocity (`obs[3]`), not just its angle, and re-run the comparison. Does it help?
3. In a text cell, answer:
   * Which agent gets closer to being "rational" in the AIMA sense, and why? (Refer to the definition: *for each possible percept sequence, select the action that maximises the expected value of the performance measure*.)
   * What is the performance measure here, precisely? Is total episode reward a good performance measure for CartPole? Why or why not?

## A Simple Learning Agent: Tabular Q-Learning

A simple reflex agent needs a human to write the rule by hand. What if, instead, the agent could work out for itself which action is best in each state, just from trial and error?

**Q-learning** is the simplest way to do this. The agent keeps a table, `Q[state, action]`, estimating the expected total future reward (performance measure) of taking `action` in `state`. After each step it nudges the table towards the reward it actually got, plus the best value it expects from the next state:

```
Q[state, action] ← Q[state, action] + alpha * (reward + gamma * max(Q[next_state]) - Q[state, action])
```

No neural network is involved — just a table of numbers, one row per state and one column per action. This only works when the environment has a small, discrete set of states, so we'll use **FrozenLake-v1**: a simple grid world where the agent must walk from a start tile to a goal tile without falling into a hole. We use the non-slippery (deterministic) version to keep things simple.

In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1", is_slippery=False)
n_states = env.observation_space.n
n_actions = env.action_space.n
Q = np.zeros((n_states, n_actions))

alpha = 0.8    # learning rate
gamma = 0.95   # how much the agent values future reward
n_episodes = 3000

rng = np.random.default_rng(0)

for ep in range(n_episodes):
    epsilon = max(0.05, 1.0 - ep / 1500)  # explore a lot at first, less later
    state, info = env.reset(seed=ep)
    done = False
    while not done:
        if rng.random() < epsilon:
            action = env.action_space.sample()          # explore
        else:
            action = int(np.argmax(Q[state]))            # exploit current knowledge
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        Q[state, action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state, action])
        state = next_state

def run_policy(policy_fn, n_episodes=100):
    successes = 0
    for ep in range(n_episodes):
        state, info = env.reset(seed=5000 + ep)
        done = False
        total_reward = 0.0
        while not done:
            action = policy_fn(state)
            state, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        if total_reward > 0:
            successes += 1
    return successes / n_episodes

random_success = run_policy(lambda s: env.action_space.sample())
qlearn_success = run_policy(lambda s: int(np.argmax(Q[s])))

print(f"Random agent success rate:      {random_success:.0%}")
print(f"Q-learning agent success rate:  {qlearn_success:.0%}")


Random agent success rate:      2%
Q-learning agent success rate:  100%


## Lab Task 4 — Experiment with Q-learning

1. Run the Q-learning cell above and record the success rate of both agents.
2. Re-run it with `is_slippery=True` (a stochastic version of FrozenLake — the agent sometimes slides in an unintended direction). What happens to the success rate? Why does this connect to the "deterministic vs. stochastic" classification from Lab Task 2?
3. In a text cell, answer:
   * Is the Q-learning agent a simple reflex agent, or something else? Justify using the agent's inputs (does it use only the current percept, or something more)?
   * Why can't we simply write a `Q` table for `CartPole-v1` the way we did for FrozenLake? (Hint: look again at CartPole's observation space.)

## Lab Submission (Unassessed but compulsory)

* **Task 1.** PEAS write-up for `MountainCar-v0` and `CartPole-v1` (text cells).
* **Task 2.** Task-environment classification table for the same two environments (text cell).
* **Task 3.** Run/modify the random-agent-vs-reflex-agent comparison for CartPole, and answer the reflection questions (text cell).
* **Task 4.** Run the Q-learning experiment (including the slippery variant) and answer the reflection questions (text cell).

Submit the completed notebook (with all explanations added as text cells and all code cells executed with visible output) on Blackboard by midnight, Monday 27th September.